# Qwen2.5-0.5B — trained from random weights, any language
One cell per stage. Run top to bottom. If a cell errors, fix that cell and
re-run just it and everything below it that depends on its output — you
don't need to redo earlier stages.

**Time budget:** a from-scratch 0.5B model cannot match the official
pretrained Qwen2.5-0.5B in a few hours — that model saw ~18 trillion tokens on
a GPU cluster. This notebook instead **measures your actual GPU speed with a
short calibration run**, then trains for as many steps as fit your time
budget, so you get the best result achievable in the time you have rather
than guessing a step count and running out of time or memory.

To switch language: change `DATASET` / `CONFIG_NAME` / `TEXT_COL` in the
**Pretrain config** cell, or in the **Continue on new language** cell.

## 1. Install

In [1]:
!pip install -q torch transformers datasets accelerate

## 2. Imports & constants
Single GPU is forced on purpose: with 2 visible GPUs, Hugging Face `Trainer`
falls back to naive `DataParallel`, which has heavy per-step overhead and was
the main cause of the original 200-hour ETA. `expandable_segments` reduces
CUDA allocator fragmentation, which caused several of the earlier OOM errors.
**These env vars only take effect on a fresh kernel — if you've run training
in this session before, restart the kernel now before running this cell.**

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, itertools, math, random, time
import torch
from datasets import load_dataset
from transformers import (AutoConfig, AutoModelForCausalLM, AutoTokenizer,
                          Trainer, TrainingArguments)

ARCH = "Qwen/Qwen2.5-0.5B"            # architecture config + tokenizer ONLY (no weights loaded from here)
REF  = "Qwen/Qwen2.5-0.5B"            # pretrained reference model used only in the compare step
MASK_TOKENS = ["<|mask_prefix|>", "<|mask_suffix|>", "<|mask_middle|>"]
EOT = "<|endoftext|>"

DEV = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
print("Device:", DEV, "| bf16:", USE_BF16, "| fp16:", USE_FP16)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def free_gpu(*names):
    """Delete named globals and release GPU memory. Call before building a new model."""
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

Device: cuda | bf16: True | fp16: False
GPU: Tesla T4


## 2b. Check the GPU
Confirm it shows close to 0% memory used right after a kernel restart. If it
already shows memory in use here, something from a previous session is still
attached — restart the kernel again before continuing.

In [3]:
!nvidia-smi

Sun Oct  4 05:44:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             27W /   70W |     107MiB /  15360MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Tokenizer and random-weight model builder

In [4]:
def get_tokenizer(path=ARCH):
    tok = AutoTokenizer.from_pretrained(path)
    tok.add_tokens(MASK_TOKENS, special_tokens=True)   # no-op if already present
    return tok


def build_model(tok, untie=False):
    """Random weights: only the architecture is taken from the Qwen2.5 config.
    AutoModelForCausalLM.from_config never downloads or loads pretrained weights."""
    cfg = AutoConfig.from_pretrained(ARCH)
    if untie:
        cfg.tie_word_embeddings = False                # separate input / output embeddings
    cfg.vocab_size = max(cfg.vocab_size, len(tok))
    cfg.use_cache = False
    model = AutoModelForCausalLM.from_config(cfg)
    n = sum(p.numel() for p in model.parameters())
    print(f"Random-init model: {n/1e9:.3f}B parameters")
    return model

## 4. Data pipeline (language-agnostic)
Tokenizes, concatenates with `<|endoftext|>`, cuts into fixed-length blocks.
A share of blocks (`mask_rate`) become masked-span infill examples instead of
plain next-token blocks, so the model also practices predicting a hidden span
from both sides of the context (Qwen2.5 is a causal decoder, so this is done as
`<mask_prefix> prefix <mask_suffix> suffix <mask_middle> hidden_span`, not BERT-style masking).

In [5]:
class PackedStream(torch.utils.data.IterableDataset):
    def __init__(self, texts, tok, col, block, mask_rate=0.0, span_max=8):
        self.texts, self.tok, self.col = texts, tok, col
        self.block, self.mask_rate, self.span_max = block, mask_rate, span_max
        self.eot = tok.convert_tokens_to_ids(EOT)
        self.P, self.S, self.M = (tok.convert_tokens_to_ids(t) for t in MASK_TOKENS)

    def _maybe_mask(self, chunk):
        if random.random() >= self.mask_rate:
            return chunk
        body = chunk[: self.block - 3]
        n = random.randint(1, self.span_max)
        s = random.randint(1, len(body) - n - 1)
        pre, mid, suf = body[:s], body[s:s + n], body[s + n:]
        return [self.P] + pre + [self.S] + suf + [self.M] + mid   # length == block

    def __iter__(self):
        buf = []
        for ex in self.texts:
            text = ex[self.col]
            if not text:
                continue
            buf += self.tok(text, add_special_tokens=False)["input_ids"] + [self.eot]
            while len(buf) >= self.block:
                chunk, buf = buf[: self.block], buf[self.block:]
                chunk = self._maybe_mask(chunk)
                yield {"input_ids": chunk, "labels": list(chunk)}


def load_text_stream(dataset, config_name, split, eval_docs, seed):
    ds = load_dataset(dataset, config_name, split=split, streaming=True)
    held_out = ds.take(eval_docs)                 # first N docs -> evaluation, never trained on
    train = ds.skip(eval_docs).shuffle(seed=seed, buffer_size=10_000)
    return train, held_out


def eval_blocks(tok, held_out, text_col, block, eval_blocks_n, mask_rate=0.0):
    ds = PackedStream(held_out, tok, text_col, block, mask_rate)
    return list(itertools.islice(iter(ds), eval_blocks_n))

## 5. Pretrain base settings
`BLOCK`, `BS`, `ACCUM` are kept small because Qwen2.5's ~152K vocabulary makes
the logits tensor (batch x block x vocab) the real memory bottleneck on a
15GB T4, not the model itself. These values are known to fit; the calibration
cell below only decides *how many steps* to run, not the memory-affecting
settings.

In [8]:
BASE_CFG = dict(
    DATASET      = "Salesforce/wikitext",
    CONFIG_NAME  = "wikitext-2-raw-v1",   # change for a different language, e.g. wikimedia/wikipedia + "20231101.ml"
    SPLIT        = "train",
    TEXT_COL     = "text",
    BLOCK        = 256,
    EVAL_DOCS    = 500,
    EVAL_BLOCKS  = 200,
    SEED         = 0,

    OUT          = "/kaggle/working/ckpt_pretrain",
    RESUME_FROM  = None,
    BS           = 2,
    ACCUM        = 32,
    LR           = 3e-4,
    MASK_RATE    = 0.25,
    UNTIE        = False,
)
print(BASE_CFG)

{'DATASET': 'Salesforce/wikitext', 'CONFIG_NAME': 'wikitext-2-raw-v1', 'SPLIT': 'train', 'TEXT_COL': 'text', 'BLOCK': 256, 'EVAL_DOCS': 500, 'EVAL_BLOCKS': 200, 'SEED': 0, 'OUT': '/kaggle/working/ckpt_pretrain', 'RESUME_FROM': None, 'BS': 2, 'ACCUM': 32, 'LR': 0.0003, 'MASK_RATE': 0.25, 'UNTIE': False}


## 6. Calibration run
Trains for a small, fixed number of real steps, times it, and calculates how
many steps fit your remaining time budget. Takes a few minutes. Set
`TARGET_HOURS` to how long you want the **pretrain** stage to run (leave
1-2 hours of your total budget free for SFT + compare afterward).

In [9]:
TARGET_HOURS = 8.0     # <-- edit this
CALIB_STEPS = 20

free_gpu("model", "trainer", "ours", "ref")

c = BASE_CFG
tok = get_tokenizer(ARCH)
model = build_model(tok, c["UNTIE"])
model.config.use_cache = False

train_texts, held_out = load_text_stream(c["DATASET"], c["CONFIG_NAME"], c["SPLIT"], c["EVAL_DOCS"], c["SEED"])
train_ds = PackedStream(train_texts, tok, c["TEXT_COL"], c["BLOCK"], c["MASK_RATE"])

targs = TrainingArguments(
    output_dir="/kaggle/working/_calib", max_steps=CALIB_STEPS,
    per_device_train_batch_size=c["BS"], gradient_accumulation_steps=c["ACCUM"],
    learning_rate=c["LR"], lr_scheduler_type="constant",
    bf16=USE_BF16, fp16=USE_FP16, gradient_checkpointing=True,
    logging_steps=1, eval_strategy="no", save_strategy="no",
    remove_unused_columns=False, report_to="none", dataloader_num_workers=1,
)
trainer = Trainer(model=model, args=targs, train_dataset=train_ds)

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

sec_per_step = elapsed / CALIB_STEPS
STEPS = max(1, int((TARGET_HOURS * 3600) / sec_per_step))
tokens_m = STEPS * c["BS"] * c["ACCUM"] * c["BLOCK"] / 1e6

print(f"\n{sec_per_step:.2f} sec/step")
print(f"STEPS that fit in {TARGET_HOURS}h: {STEPS}")
print(f"Tokens to be seen: {tokens_m:.1f}M")

free_gpu("model", "trainer")

GPU memory free: 15.53 GB / 15.64 GB
Random-init model: 0.494B parameters


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Step,Training Loss
1,12.103140
2,11.344793
3,10.503326
4,10.191475
5,9.914766
6,9.624401
7,9.332410
8,9.073750
9,8.815284
10,8.569830



30.11 sec/step
STEPS that fit in 8.0h: 956
Tokens to be seen: 15.7M
GPU memory free: 15.44 GB / 15.64 GB


## 7. Run pretraining

In [11]:
free_gpu("model", "trainer", "ours", "ref")

c = BASE_CFG
random.seed(c["SEED"]); torch.manual_seed(c["SEED"])

tok = get_tokenizer(c["RESUME_FROM"] or ARCH)
model = (AutoModelForCausalLM.from_pretrained(c["RESUME_FROM"])
         if c["RESUME_FROM"] else build_model(tok, c["UNTIE"]))
model.config.use_cache = False
if model.get_input_embeddings().num_embeddings < len(tok):
    model.resize_token_embeddings(len(tok))

train_texts, held_out = load_text_stream(c["DATASET"], c["CONFIG_NAME"], c["SPLIT"],
                                         c["EVAL_DOCS"], c["SEED"])
train_ds = PackedStream(train_texts, tok, c["TEXT_COL"], c["BLOCK"], c["MASK_RATE"])
val = eval_blocks(tok, held_out, c["TEXT_COL"], c["BLOCK"], c["EVAL_BLOCKS"], mask_rate=0.0)

targs = TrainingArguments(
    output_dir=c["OUT"], max_steps=STEPS,      # STEPS comes from the calibration cell above
    per_device_train_batch_size=c["BS"], gradient_accumulation_steps=c["ACCUM"],
    learning_rate=c["LR"], lr_scheduler_type="cosine",warmup_steps=max(1, int(0.02 * STEPS)),
    weight_decay=0.1, adam_beta2=0.95, max_grad_norm=1.0,
    bf16=USE_BF16, fp16=USE_FP16, gradient_checkpointing=True,
    logging_steps=50, eval_strategy="steps", eval_steps=max(200, STEPS // 10),
    save_steps=max(200, STEPS // 10), save_total_limit=2,
    remove_unused_columns=False, report_to="none", dataloader_num_workers=1,
)
trainer = Trainer(model=model, args=targs, train_dataset=train_ds, eval_dataset=val)
trainer.train()
model.save_pretrained(c["OUT"]); tok.save_pretrained(c["OUT"])
print("Saved to", c["OUT"])

GPU memory free: 15.44 GB / 15.64 GB
Random-init model: 0.494B parameters


Step,Training Loss,Validation Loss
200,5.850112,5.712167
400,5.052073,5.317496
600,4.706212,5.131159
800,4.575178,5.091970
956,4.552032,5.092556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /kaggle/working/ckpt_pretrain


## 8. (Optional) Continue on a new language
Run instead of / after cell 7 once you have a checkpoint and want to adapt it
to a second language. Re-run the calibration pattern with this config if you
want a time-budgeted step count for this stage too; otherwise set `STEPS2`
directly.

In [13]:
NEW_LANG_CFG = dict(
    DATASET      = "wikimedia/wikipedia",
    CONFIG_NAME  = "20231101.ml",     # Malayalam Wikipedia dump
    SPLIT        = "train",
    TEXT_COL     = "text",
    BLOCK        = 256,
    EVAL_DOCS    = 500,
    EVAL_BLOCKS  = 200,
    SEED         = 0,

    OUT          = "/kaggle/working/ckpt_lang2",
    RESUME_FROM  = BASE_CFG["OUT"],   # continue from the checkpoint saved in cell 7
    BS           = 2,
    ACCUM        = 32,
    LR           = 5e-5,       # lower than pretraining LR to avoid forgetting language 1
    MASK_RATE    = 0.25,
    UNTIE        = False,
)
STEPS2 = 1000   # edit directly, or re-run the calibration pattern from cell 6 with NEW_LANG_CFG

free_gpu("model", "trainer", "ours", "ref")

c = NEW_LANG_CFG
random.seed(c["SEED"]); torch.manual_seed(c["SEED"])

tok = get_tokenizer(c["RESUME_FROM"])
model = AutoModelForCausalLM.from_pretrained(c["RESUME_FROM"])
model.config.use_cache = False
if model.get_input_embeddings().num_embeddings < len(tok):
    model.resize_token_embeddings(len(tok))

train_texts, held_out = load_text_stream(c["DATASET"], c["CONFIG_NAME"], c["SPLIT"],
                                         c["EVAL_DOCS"], c["SEED"])
train_ds = PackedStream(train_texts, tok, c["TEXT_COL"], c["BLOCK"], c["MASK_RATE"])
val = eval_blocks(tok, held_out, c["TEXT_COL"], c["BLOCK"], c["EVAL_BLOCKS"], mask_rate=0.0)

targs = TrainingArguments(
    output_dir=c["OUT"], max_steps=STEPS2,
    per_device_train_batch_size=c["BS"], gradient_accumulation_steps=c["ACCUM"],
    learning_rate=c["LR"], lr_scheduler_type="cosine", warmup_steps=max(1, int(0.02 * STEPS2)),
    weight_decay=0.1, adam_beta2=0.95, max_grad_norm=1.0,
    bf16=USE_BF16, fp16=USE_FP16, gradient_checkpointing=True,
    logging_steps=50, eval_strategy="steps", eval_steps=max(200, STEPS2 // 10),
    save_steps=max(200, STEPS2 // 10), save_total_limit=2,
    remove_unused_columns=False, report_to="none", dataloader_num_workers=1,
)
trainer = Trainer(model=model, args=targs, train_dataset=train_ds, eval_dataset=val)
trainer.train()
model.save_pretrained(c["OUT"]); tok.save_pretrained(c["OUT"])
print("Saved to", c["OUT"])

GPU memory free: 15.44 GB / 15.64 GB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 9. Instruction fine-tuning (SFT) — config
Loss is computed on the response tokens only (the instruction/prompt part is
masked with -100). Kept to the same small `BLOCK`/`BS` pattern for memory safety.

In [14]:
SFT_CFG = dict(
    MODEL        = BASE_CFG["OUT"],   # or NEW_LANG_CFG["OUT"] for the second-language model
    DATASET      = "tatsu-lab/alpaca",
    CONFIG_NAME  = None,
    SPLIT        = "train",
    INST_COL     = "instruction",
    INPUT_COL    = "input",
    RESP_COL     = "output",
    MAX_EXAMPLES = 2000,     # raise for a longer run if time allows
    BLOCK        = 256,

    OUT          = "/kaggle/working/ckpt_sft",
    EPOCHS       = 2,
    BS           = 2,
    ACCUM        = 16,
    LR           = 2e-5,
)
PROMPT = "### Instruction:\n{ins}\n\n### Response:\n"
print(SFT_CFG)

{'MODEL': '/kaggle/working/ckpt_pretrain', 'DATASET': 'tatsu-lab/alpaca', 'CONFIG_NAME': None, 'SPLIT': 'train', 'INST_COL': 'instruction', 'INPUT_COL': 'input', 'RESP_COL': 'output', 'MAX_EXAMPLES': 2000, 'BLOCK': 256, 'OUT': '/kaggle/working/ckpt_sft', 'EPOCHS': 2, 'BS': 2, 'ACCUM': 16, 'LR': 2e-05}


## 10. Run instruction fine-tuning

In [16]:
free_gpu("model", "trainer", "ours", "ref")

c = SFT_CFG
tok = get_tokenizer(c["MODEL"])
model = AutoModelForCausalLM.from_pretrained(c["MODEL"])
model.config.use_cache = False
eot = tok.convert_tokens_to_ids(EOT)

raw = load_dataset(c["DATASET"], c["CONFIG_NAME"], split=c["SPLIT"])
raw = raw.select(range(min(len(raw), c["MAX_EXAMPLES"])))

def encode(ex):
    ins = ex[c["INST_COL"]] + (("\n\n" + ex[c["INPUT_COL"]])
                               if c["INPUT_COL"] and ex.get(c["INPUT_COL"]) else "")
    p = tok(PROMPT.format(ins=ins), add_special_tokens=False)["input_ids"]
    r = tok(ex[c["RESP_COL"]], add_special_tokens=False)["input_ids"] + [eot]
    ids = (p + r)[: c["BLOCK"]]
    labels = ([-100] * len(p) + r)[: c["BLOCK"]]     # loss on the response only
    return {"input_ids": ids, "labels": labels}

data = raw.map(encode, remove_columns=raw.column_names)

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = eot
    ids = torch.tensor([b["input_ids"] + [pad] * (m - len(b["input_ids"])) for b in batch])
    lab = torch.tensor([b["labels"] + [-100] * (m - len(b["labels"])) for b in batch])
    att = torch.tensor([[1]*len(b["input_ids"]) + [0]*(m-len(b["input_ids"])) for b in batch])
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

targs = TrainingArguments(
    output_dir=c["OUT"], num_train_epochs=c["EPOCHS"],
    per_device_train_batch_size=c["BS"], gradient_accumulation_steps=c["ACCUM"],
    learning_rate=c["LR"], lr_scheduler_type="cosine", warmup_steps=20,
    bf16=USE_BF16, fp16=USE_FP16, gradient_checkpointing=True, logging_steps=25, save_strategy="no",
    remove_unused_columns=False, report_to="none", dataloader_num_workers=1,
)
trainer = Trainer(model=model, args=targs, train_dataset=data, data_collator=collate)
trainer.train()
model.save_pretrained(c["OUT"]); tok.save_pretrained(c["OUT"])
print("Saved to", c["OUT"])

GPU memory free: 15.44 GB / 15.64 GB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
25,8.037034
50,7.122929
75,6.755664
100,6.666136
125,6.808552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /kaggle/working/ckpt_sft


## 11. Comparison functions

In [17]:
@torch.no_grad()
def perplexity(model, blocks, bs=2):
    model.eval().to(DEV)
    tot_loss, tot_tok = 0.0, 0
    for i in range(0, len(blocks), bs):
        ids = torch.tensor([b["input_ids"] for b in blocks[i:i+bs]]).to(DEV)
        with torch.autocast(DEV, dtype=(torch.bfloat16 if USE_BF16 else torch.float16), enabled=DEV == "cuda"):
            logits = model(input_ids=ids).logits[:, :-1].float()
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1), reduction="sum")
        tot_loss += loss.item(); tot_tok += ids[:, 1:].numel()
    model.to("cpu")
    return math.exp(tot_loss / tot_tok)


@torch.no_grad()
def mask_accuracy(model, tok, blocks, n=100, span=3):
    P, S, M = (tok.convert_tokens_to_ids(t) for t in MASK_TOKENS)
    model.eval().to(DEV); hit = 0; cnt = 0
    rng = random.Random(0)
    for b in blocks[:n]:
        body = b["input_ids"][:200]
        if len(body) < span + 40:
            continue
        s = rng.randint(20, len(body) - span - 20)
        prompt = [P] + body[:s] + [S] + body[s+span:] + [M]
        out = model.generate(torch.tensor([prompt]).to(DEV), max_new_tokens=span,
                             do_sample=False, use_cache=True)[0, len(prompt):].tolist()
        hit += out == body[s:s+span]; cnt += 1
    model.to("cpu")
    return hit / max(cnt, 1)


def sample(model, tok, prompt, n=40):
    model.eval().to(DEV)
    ids = tok(prompt, return_tensors="pt").input_ids.to(DEV)
    out = model.generate(ids, max_new_tokens=n, do_sample=False, use_cache=True,
                         repetition_penalty=1.1)
    text = tok.decode(out[0], skip_special_tokens=True)
    model.to("cpu")
    return text

## 12. Compare config and run
Loads and evaluates **one model at a time**, moving each to CPU when done,
so `ours` and `ref` never both sit on the GPU at once — this avoids the OOM
you'd otherwise get from holding two 0.6B models plus logits tensors together
on a 15GB T4.

In [18]:
COMPARE_CFG = dict(
    MODEL   = SFT_CFG["OUT"],       # model to evaluate: pretrain, new-language, or SFT checkpoint
    REF     = REF,                  # official pretrained Qwen2.5-0.5B
    DATASET = BASE_CFG["DATASET"],
    CONFIG_NAME = BASE_CFG["CONFIG_NAME"],
    SPLIT   = "train",
    TEXT_COL = "text",
    BLOCK   = 256,
    EVAL_DOCS = 500,
    EVAL_BLOCKS = 200,
    SEED    = 0,
    PROMPTS = ["The capital of France is", "In machine learning, overfitting means"],
)
print(COMPARE_CFG)

{'MODEL': '/kaggle/working/ckpt_sft', 'REF': 'Qwen/Qwen2.5-0.5B', 'DATASET': 'Salesforce/wikitext', 'CONFIG_NAME': 'wikitext-2-raw-v1', 'SPLIT': 'train', 'TEXT_COL': 'text', 'BLOCK': 256, 'EVAL_DOCS': 500, 'EVAL_BLOCKS': 200, 'SEED': 0, 'PROMPTS': ['The capital of France is', 'In machine learning, overfitting means']}


In [19]:
free_gpu("model", "trainer", "ours", "ref")

c = COMPARE_CFG
tok = get_tokenizer(c["MODEL"])
_, held_out = load_text_stream(c["DATASET"], c["CONFIG_NAME"], c["SPLIT"], c["EVAL_DOCS"], c["SEED"])
blocks = eval_blocks(tok, held_out, c["TEXT_COL"], c["BLOCK"], c["EVAL_BLOCKS"], mask_rate=0.0)

ours = AutoModelForCausalLM.from_pretrained(c["MODEL"])
ppl_o = perplexity(ours, blocks)
mask_acc = mask_accuracy(ours, tok, blocks)
samples_ours = [sample(ours, tok, p) for p in c["PROMPTS"]]
del ours; free_gpu()

ref = AutoModelForCausalLM.from_pretrained(c["REF"])
ppl_r = perplexity(ref, blocks)
samples_ref = [sample(ref, tok, p) for p in c["PROMPTS"]]
del ref; free_gpu()

print(f"\nPerplexity on held-out text ({len(blocks)} blocks of {c['BLOCK']} tokens)")
print(f"  ours      : {ppl_o:8.2f}")
print(f"  pretrained: {ppl_r:8.2f}   (gap: {ppl_o/ppl_r:.2f}x)")
print(f"Masked-span exact-match accuracy (ours): {mask_acc:.3f}")

for p, so, sr in zip(c["PROMPTS"], samples_ours, samples_ref):
    print("\nPROMPT:", p)
    print("  ours      :", so)
    print("  pretrained:", sr)

GPU memory free: 15.44 GB / 15.64 GB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

GPU memory free: 15.44 GB / 15.64 GB


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

GPU memory free: 15.44 GB / 15.64 GB

Perplexity on held-out text (109 blocks of 256 tokens)
  ours      :   170.49
  pretrained:    28.03   (gap: 6.08x)
Masked-span exact-match accuracy (ours): 0.000

PROMPT: The capital of France is
  ours      : The capital of France is a number of the city 's main area of the 19th century . 

  pretrained: The capital of France is Paris. The population of the city is 2,180,347 as of 2015. It has a total area of 609 km² and a

PROMPT: In machine learning, overfitting means
  ours      : In machine learning, overfitting means of the game 's " The game 's most important and more than a good and a lot of people who have been used to be a great @-@ time " . 

  pretrained: In machine learning, overfitting means that a model is too complex and fits the training data too closely. This can lead to poor performance on new, unseen data. How can we measure the complexity of a model in terms of its capacity


## 13. (Optional) Standard benchmark comparison
```
!pip install -q lm-eval
!lm_eval --model hf --model_args pretrained={SFT_CFG['OUT']} --tasks hellaswag,arc_easy,piqa --batch_size 8
!lm_eval --model hf --model_args pretrained={REF} --tasks hellaswag,arc_easy,piqa --batch_size 8
```